# 08 — Marker-based annotation on the canonical 18-cluster article branch

This notebook reproduces the archived marker-based annotation step while
remaining safe on the current Windows / 16-GB-RAM system.

## Canonical scientific input

The input is the notebook-06 canonical full-gene object:

- 351,427 cells;
- 15,176 genes;
- `leiden_scvi_main` = historical article-defining `r=0.60`;
- 18 clusters;
- raw counts preserved in `layers["counts"]`;
- canonical historical `X_scVI` and `X_umap`.

The marker evidence tables come from notebook `07`, whose historical
Wilcoxon checkpoints have already been reproduced.

## Historical annotation semantics preserved

The archived executed notebook used:

- the same marker panels;
- the same top-100 / top-20 DE evidence logic;
- the same conservative manual 18-cluster annotation map;
- the same bias/caution rules;
- the principle that tumor-like labels remain **putative until CNV validation**;
- the same marker-panel dotplots, based on a small raw-count gene subset.

Only the internal execution is changed to avoid loading the 15,176-gene H5AD
into memory. The final annotated H5AD is produced by sequentially copying the
canonical notebook-06 backbone and patching `/obs` plus annotation provenance.

The two old dataset identifiers in free-text notes are updated to the final
project nomenclature (`DS5 -> DS3`, `DS13 -> DS4`); biological assignments are
unchanged.

## Archived execution checkpoints

The historical executed notebook reported:

```text
input AnnData                 351427 × 15176
clusters                     18
marker DE top100             (3600, 12)
marker DE top20              (360, 12)
cluster QC                   (18, 21)
cluster composition          (2538, 5)
marker-panel presence input  (81, 3)

annotation panel summary     (16, 6)
annotation table             (18, 6)
annotation evidence          (18, 53)
bias flags                   (18, 29)
annotation cell counts       (18, 8)
annotated composition        (2538, 10)
decision report              (15, 2)

plot marker genes present    59 / 59
```

The current standardized DS1–DS4 nomenclature is retained in all new outputs.

In [1]:
# =========================
# 0) IMPORTS / PATHS / PARAMETERS
# =========================

import os
import gc
import shutil
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

try:
    from anndata.io import read_elem, write_elem
except ImportError:
    from anndata._io.specs import read_elem, write_elem


warnings.filterwarnings("ignore")


def _find_repository_root(start=None):
    start = (
        Path.cwd()
        if start is None
        else Path(start)
    )
    start = start.expanduser().resolve()

    for candidate in (
        start,
        *start.parents,
    ):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        _find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

INTEGRATED_DIR = (
    PROCESSED_DIR
    / "integrated"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

for directory in (
    PROCESSED_DIR,
    INTEGRATED_DIR,
    METADATA_DIR,
    FIGURES_DIR,
    RESULTS_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Canonical notebook-06 full-gene backbone.
FULLGENE_INPUT_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad"
)


# Notebook-07 exact historical DE tables.
MARKER_DE_TOP100_TSV = (
    METADATA_DIR
    / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top100_up_down.tsv"
)

MARKER_DE_TOP20_TSV = (
    METADATA_DIR
    / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top20_up.tsv"
)

CLUSTER_QC_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_cluster_QC_summary.tsv"
)

CLUSTER_COMPOSITION_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_cluster_composition_summary.tsv"
)

MARKER_PANEL_PRESENCE_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_marker_panel_presence.tsv"
)


# Annotation outputs.
ANNOTATION_TSV = (
    METADATA_DIR
    / "annotation_marker_based_HVG8000_r0_6.tsv"
)

ANNOTATION_EVIDENCE_TSV = (
    METADATA_DIR
    / "cluster_marker_annotation_evidence_HVG8000_r0_6.tsv"
)

ANNOTATION_COUNTS_TSV = (
    METADATA_DIR
    / "annotation_marker_based_HVG8000_r0_6_cell_counts.tsv"
)

ANNOTATION_COMPOSITION_TSV = (
    METADATA_DIR
    / "annotation_marker_based_HVG8000_r0_6_composition_annotated.tsv"
)

ANNOTATION_BIAS_TSV = (
    METADATA_DIR
    / "annotation_marker_based_HVG8000_r0_6_bias_flags.tsv"
)

ANNOTATION_DECISION_REPORT_TSV = (
    METADATA_DIR
    / "annotation_marker_based_HVG8000_r0_6_decision_report.tsv"
)

ANNOTATION_OUTPUT_MANIFEST_TSV = (
    METADATA_DIR
    / "annotation_marker_based_HVG8000_r0_6_output_manifest.tsv"
)


ANNOTATED_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_based_annotation.h5ad"
)

ANNOTATED_H5AD_ALIAS = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation.h5ad"
)

ANNOTATED_BUILDING_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_based_annotation.building.h5ad"
)


FIG_DIR = (
    FIGURES_DIR
    / "annotation_marker_based_HVG8000_r0_6"
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


CLUSTER_KEY = (
    "leiden_scvi_main"
)

SOURCE_CLUSTER_KEY = (
    "leiden_scvi_HVG8000_r0_6"
)

ANNOTATION_COL = (
    "annotation_marker_based_HVG8000_r0_6"
)

LINEAGE_COL = (
    "lineage_marker_based_HVG8000_r0_6"
)

CONFIDENCE_COL = (
    "annotation_confidence_HVG8000_r0_6"
)

MALIGNANCY_COL = (
    "malignancy_status_marker_based_HVG8000_r0_6"
)

CAUTION_COL = (
    "annotation_caution_HVG8000_r0_6"
)

NOTE_COL = (
    "annotation_note_HVG8000_r0_6"
)


EXPECTED_N_CELLS = 351_427
EXPECTED_N_GENES = 15_176
EXPECTED_N_CLUSTERS = 18

EXPECTED_CLUSTER_MIN = 105
EXPECTED_CLUSTER_MAX = 61_646
EXPECTED_CLUSTER_MEDIAN = 14_336.5

EXPECTED_MARKER_TOP100_SHAPE = (
    3_600,
    12,
)

EXPECTED_MARKER_TOP20_SHAPE = (
    360,
    12,
)

EXPECTED_CLUSTER_QC_SHAPE = (
    18,
    21,
)

EXPECTED_CLUSTER_COMPOSITION_SHAPE = (
    2_538,
    5,
)

EXPECTED_MARKER_PANEL_INPUT_SHAPE = (
    81,
    3,
)

EXPECTED_PANEL_SUMMARY_SHAPE = (
    16,
    6,
)

EXPECTED_ANNOTATION_TABLE_SHAPE = (
    18,
    6,
)

EXPECTED_EVIDENCE_SHAPE = (
    18,
    53,
)

EXPECTED_BIAS_SHAPE = (
    18,
    29,
)

EXPECTED_COUNTS_SHAPE = (
    18,
    8,
)

EXPECTED_ANNOTATED_COMPOSITION_SHAPE = (
    2_538,
    10,
)

EXPECTED_DECISION_REPORT_SHAPE = (
    15,
    2,
)

EXPECTED_PLOT_MARKER_GENES = 59


# Sequential HDF5 scan size for the optional 59-gene historical dotplot.
MAX_NNZ_PER_ROW_BLOCK = 8_000_000


sc.settings.figdir = str(
    FIG_DIR
)

sc.settings.verbosity = 2


def write_tsv_replace(
    dataframe,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    dataframe.to_csv(
        path,
        sep="\t",
        index=False,
    )

    print(
        f"[SAVED] {path} "
        f"shape={dataframe.shape}"
    )


def safe_read_tsv(
    path,
    required=True,
):
    path = Path(path)

    if not path.exists():
        if required:
            raise FileNotFoundError(
                path
            )

        print(
            f"[WARNING] missing optional file: {path}"
        )

        return pd.DataFrame()

    dataframe = pd.read_csv(
        path,
        sep="\t",
    )

    print(
        f"[READ] {path} "
        f"shape={dataframe.shape}"
    )

    return dataframe


def present_genes(
    genes,
    var_names,
):
    var_set_local = set(
        map(
            str,
            var_names,
        )
    )

    return [
        gene
        for gene
        in genes
        if gene
        in var_set_local
    ]


def get_cluster_order(
    labels,
):
    values = (
        pd.Series(
            labels
        )
        .astype(str)
        .unique()
        .tolist()
    )

    def keyfun(value):
        return (
            0,
            int(value),
        ) if str(
            value
        ).isdigit() else (
            1,
            str(value),
        )

    return sorted(
        values,
        key=keyfun,
    )


def stored_shape(
    node,
):
    if isinstance(
        node,
        h5py.Dataset,
    ):
        return tuple(
            int(x)
            for x
            in node.shape
        )

    shape = node.attrs.get(
        "shape"
    )

    if shape is None:
        raise KeyError(
            f"{node.name} has no stored shape."
        )

    return tuple(
        int(x)
        for x
        in shape
    )


def replace_h5_key(
    parent,
    key,
    value,
):
    if key in parent:
        del parent[
            key
        ]

    write_elem(
        parent,
        key,
        value,
    )


print(
    "Project:",
    PROJECT_DIR,
)

print(
    "Input h5ad:",
    FULLGENE_INPUT_H5AD,
)

print(
    "Cluster key:",
    CLUSTER_KEY,
)

Project: G:\Repository\glioma-cnv-single-cell-atlas
Input h5ad: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad
Cluster key: leiden_scvi_main


In [2]:
# =========================
# 1) METADATA-ONLY INPUT VALIDATION
# =========================

if not FULLGENE_INPUT_H5AD.exists():
    raise FileNotFoundError(
        FULLGENE_INPUT_H5AD
    )


with h5py.File(
    FULLGENE_INPUT_H5AD,
    "r",
) as handle:

    if "obs" not in handle:
        raise KeyError(
            "Input H5AD is missing /obs."
        )

    if "var" not in handle:
        raise KeyError(
            "Input H5AD is missing /var."
        )

    if (
        "layers" not in handle
        or "counts"
        not in handle[
            "layers"
        ]
    ):
        raise KeyError(
            "Input H5AD is missing layers['counts']."
        )

    obs = read_elem(
        handle[
            "obs"
        ]
    )

    var = read_elem(
        handle[
            "var"
        ]
    )

    x_shape = stored_shape(
        handle[
            "X"
        ]
    )

    counts_shape = stored_shape(
        handle[
            "layers"
        ][
            "counts"
        ]
    )

    counts_encoding = (
        handle[
            "layers"
        ][
            "counts"
        ].attrs.get(
            "encoding-type",
            "",
        )
    )

    if isinstance(
        counts_encoding,
        bytes,
    ):
        counts_encoding = (
            counts_encoding.decode(
                "utf-8"
            )
        )

    x_umap = (
        np.asarray(
            handle[
                "obsm"
            ][
                "X_umap"
            ],
            dtype=np.float32,
        )
        if (
            "obsm" in handle
            and "X_umap"
            in handle[
                "obsm"
            ]
        )
        else None
    )


if x_shape != (
    EXPECTED_N_CELLS,
    EXPECTED_N_GENES,
):
    raise AssertionError(
        "Unexpected input X shape: "
        f"{x_shape}"
    )

if counts_shape != x_shape:
    raise AssertionError(
        "X and layers['counts'] shapes differ."
    )

if counts_encoding != "csr_matrix":
    raise AssertionError(
        "Expected CSR counts layer."
    )

if len(
    obs
) != EXPECTED_N_CELLS:
    raise AssertionError(
        "Unexpected cell count."
    )

if len(
    var
) != EXPECTED_N_GENES:
    raise AssertionError(
        "Unexpected gene count."
    )

if not var.index.is_unique:
    raise AssertionError(
        "var_names are not unique."
    )


if CLUSTER_KEY not in obs.columns:
    if SOURCE_CLUSTER_KEY in obs.columns:
        obs[
            CLUSTER_KEY
        ] = (
            obs[
                SOURCE_CLUSTER_KEY
            ]
            .astype(str)
            .astype(
                "category"
            )
        )

        print(
            f"[ADDED] {CLUSTER_KEY} "
            f"from {SOURCE_CLUSTER_KEY}"
        )

    else:
        raise KeyError(
            f"Neither {CLUSTER_KEY} nor "
            f"{SOURCE_CLUSTER_KEY} exists in obs."
        )


obs[
    CLUSTER_KEY
] = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .astype(
        "category"
    )
)


cluster_order = get_cluster_order(
    obs[
        CLUSTER_KEY
    ]
)


cluster_counts = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .value_counts()
)


if len(
    cluster_order
) != EXPECTED_N_CLUSTERS:
    raise AssertionError(
        "Canonical annotation input is not 18 clusters."
    )

if int(
    cluster_counts.min()
) != EXPECTED_CLUSTER_MIN:
    raise AssertionError(
        "Canonical minimum cluster size changed."
    )

if int(
    cluster_counts.max()
) != EXPECTED_CLUSTER_MAX:
    raise AssertionError(
        "Canonical maximum cluster size changed."
    )

if not np.isclose(
    float(
        cluster_counts.median()
    ),
    EXPECTED_CLUSTER_MEDIAN,
):
    raise AssertionError(
        "Canonical median cluster size changed."
    )


expected_datasets = {
    "DS1_GSE103224",
    "DS2_GSE141383",
    "DS3_GSE182109",
    "DS4_GSE278450",
}

observed_datasets = set(
    obs[
        "core_dataset"
    ]
    .astype(str)
    .unique()
)

if observed_datasets != expected_datasets:
    raise AssertionError(
        "Input core_dataset nomenclature is not "
        "the final DS1/DS2/DS3/DS4 scheme."
    )


print(
    "Validated shape:",
    x_shape,
)

print(
    "n clusters:",
    len(
        cluster_order
    ),
)

print(
    "clusters:",
    cluster_order,
)

print(
    "counts layer:",
    counts_shape,
)

print(
    "X_umap:",
    None
    if x_umap is None
    else x_umap.shape,
)

print(
    "core datasets:",
    sorted(
        observed_datasets
    ),
)


# Notebook-07 historical DE/QC outputs.
marker_de = safe_read_tsv(
    MARKER_DE_TOP100_TSV,
    required=True,
)

marker_top20 = safe_read_tsv(
    MARKER_DE_TOP20_TSV,
    required=True,
)

cluster_qc = safe_read_tsv(
    CLUSTER_QC_TSV,
    required=True,
)

cluster_comp = safe_read_tsv(
    CLUSTER_COMPOSITION_TSV,
    required=True,
)

marker_panel_presence = safe_read_tsv(
    MARKER_PANEL_PRESENCE_TSV,
    required=True,
)


expected_shapes = {
    "marker_de":
        (
            marker_de.shape,
            EXPECTED_MARKER_TOP100_SHAPE,
        ),

    "marker_top20":
        (
            marker_top20.shape,
            EXPECTED_MARKER_TOP20_SHAPE,
        ),

    "cluster_qc":
        (
            cluster_qc.shape,
            EXPECTED_CLUSTER_QC_SHAPE,
        ),

    "cluster_comp":
        (
            cluster_comp.shape,
            EXPECTED_CLUSTER_COMPOSITION_SHAPE,
        ),

    "marker_panel_presence":
        (
            marker_panel_presence.shape,
            EXPECTED_MARKER_PANEL_INPUT_SHAPE,
        ),
}


for name, (
    observed_shape,
    expected_shape,
) in expected_shapes.items():
    if observed_shape != expected_shape:
        raise AssertionError(
            f"{name} shape changed: "
            f"{observed_shape} vs {expected_shape}"
        )


for dataframe in (
    marker_de,
    marker_top20,
    cluster_qc,
    cluster_comp,
):
    dataframe[
        "cluster"
    ] = (
        dataframe[
            "cluster"
        ]
        .astype(str)
    )


var_names = pd.Index(
    var.index.astype(str)
)

var_set = set(
    var_names
)


print(
    "\n[HISTORICAL NOTEBOOK-07 INPUT CHECKPOINTS REPRODUCED]"
)

Validated shape: (351427, 15176)
n clusters: 18
clusters: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17']
counts layer: (351427, 15176)
X_umap: (351427, 2)
core datasets: ['DS1_GSE103224', 'DS2_GSE141383', 'DS3_GSE182109', 'DS4_GSE278450']
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top100_up_down.tsv shape=(3600, 12)
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top20_up.tsv shape=(360, 12)
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_cluster_QC_summary.tsv shape=(18, 21)
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_cluster_composition_summary.tsv shape=(2538, 5)
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_marker_panel_presence.tsv shape=(81, 3)

[HISTORICAL NOTEBOOK-07 INPUT CHECKPOINTS REPRODUCED]


In [3]:
# =========================
# Marker panels
# =========================

MARKER_PANELS = {
    "Tumor_AC_like": ["GFAP", "AQP4", "ALDH1L1", "SLC1A3", "APOE", "CLU", "SPARCL1", "SLC1A2", "FABP7"],
    "Tumor_OPC_like": ["PDGFRA", "OLIG1", "OLIG2", "SOX10", "CSPG4", "BCAN", "PTPRZ1", "PTN", "DLL3"],
    "Tumor_NPC_like": ["SOX4", "SOX11", "NES", "ASCL1", "DCX", "STMN2", "TUBB3", "MYT1L", "DLX5"],
    "Tumor_MES_like": ["CD44", "CHI3L1", "SERPINE1", "VIM", "FN1", "TGFBI", "ANXA1", "LGALS3", "CTHRC1"],
    "Hypoxia_stress": ["CA9", "VEGFA", "HILPDA", "LOX", "BNIP3", "NDRG1", "ADM", "DDIT4"],
    "Cycling": ["MKI67", "TOP2A", "UBE2C", "BIRC5", "PBK", "CENPF", "NUSAP1", "CDK1"],
    "Myeloid_TAM": ["LYZ", "TYROBP", "AIF1", "C1QA", "C1QB", "C1QC", "TREM2", "CSF1R", "APOE", "CD68", "CD163"],
    "Inflammatory_myeloid": ["IL1B", "CCL3", "CCL4", "S100A8", "S100A9", "FCN1", "AQP9", "FPR2", "LILRA5"],
    "T_NK": ["CD3D", "CD3E", "TRAC", "LCK", "NKG7", "GNLY", "GZMB", "PRF1", "KLRD1"],
    "B_Plasma": ["CD79A", "CD79B", "MS4A1", "MZB1", "JCHAIN", "IGHM", "IGKC", "POU2AF1", "SDC1"],
    "Oligodendrocyte": ["MOG", "MAG", "MOBP", "PLP1", "MBP", "ERMN", "CLDN11"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "FLT1", "ESAM", "EMCN", "CLEC14A", "ERG"],
    "Mural_Pericyte": ["PDGFRB", "ACTA2", "RGS5", "MCAM", "CSPG4", "NOTCH3", "ENPEP", "ABCC9"],
    "Fibroblast_ECM": ["COL1A1", "COL1A2", "COL3A1", "LUM", "DCN", "CTHRC1", "COL6A1", "COL6A2"],
    "Erythroid_Hb": ["HBA1", "HBA2", "HBB", "ALAS2"],
    "Mast": ["KIT", "TPSAB1", "TPSB2", "CPA3", "MS4A2", "GATA2", "RGS13", "ENPP3"],
}

panel_rows = []
for panel, genes in MARKER_PANELS.items():
    present = present_genes(genes, var_names)
    missing = [g for g in genes if g not in present]
    panel_rows.append({
        "panel": panel,
        "n_panel_genes": len(genes),
        "n_present": len(present),
        "n_missing": len(missing),
        "present_genes": ";".join(present),
        "missing_genes": ";".join(missing),
    })

panel_presence_df = pd.DataFrame(panel_rows)
PANEL_PRESENCE_TSV = METADATA_DIR / "annotation_marker_panel_presence_HVG8000_r0_6.tsv"
write_tsv_replace(panel_presence_df, PANEL_PRESENCE_TSV)
display(panel_presence_df)


if panel_presence_df.shape != EXPECTED_PANEL_SUMMARY_SHAPE:
    raise AssertionError(
        "Annotation marker-panel summary shape changed: "
        f"{panel_presence_df.shape}"
    )

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\annotation_marker_panel_presence_HVG8000_r0_6.tsv shape=(16, 6)


,panel,n_panel_genes,n_present,n_missing,present_genes,missing_genes
0,Tumor_AC_like,9,9,0,GFAP;AQP4;ALDH1L1;SLC1A3;APOE;CLU;SPARCL1;SLC1...,
1,Tumor_OPC_like,9,9,0,PDGFRA;OLIG1;OLIG2;SOX10;CSPG4;BCAN;PTPRZ1;PTN...,
2,Tumor_NPC_like,9,9,0,SOX4;SOX11;NES;ASCL1;DCX;STMN2;TUBB3;MYT1L;DLX5,
3,Tumor_MES_like,9,9,0,CD44;CHI3L1;SERPINE1;VIM;FN1;TGFBI;ANXA1;LGALS...,
4,Hypoxia_stress,8,8,0,CA9;VEGFA;HILPDA;LOX;BNIP3;NDRG1;ADM;DDIT4,
5,Cycling,8,8,0,MKI67;TOP2A;UBE2C;BIRC5;PBK;CENPF;NUSAP1;CDK1,
6,Myeloid_TAM,11,11,0,LYZ;TYROBP;AIF1;C1QA;C1QB;C1QC;TREM2;CSF1R;APO...,
7,Inflammatory_myeloid,9,9,0,IL1B;CCL3;CCL4;S100A8;S100A9;FCN1;AQP9;FPR2;LI...,
8,T_NK,9,8,1,CD3D;CD3E;TRAC;LCK;NKG7;GNLY;GZMB;KLRD1,PRF1
9,B_Plasma,9,7,2,CD79A;CD79B;MZB1;IGHM;IGKC;POU2AF1;SDC1,MS4A1;JCHAIN


In [4]:
# =========================
# Summarize DE marker evidence per cluster
# =========================

required_cols = {"cluster", "gene", "direction", "logfoldchange", "padj", "pct_in_cluster", "pct_rest"}
missing_cols = required_cols - set(marker_de.columns)
if missing_cols:
    raise ValueError(f"marker_de is missing required columns: {missing_cols}")

up_de = marker_de[marker_de["direction"].astype(str).eq("up")].copy()
down_de = marker_de[marker_de["direction"].astype(str).eq("down")].copy()

# Make sure top genes are ordered by direction rank where available.
rank_col = "direction_rank" if "direction_rank" in marker_de.columns else "rank_scanpy"
up_de = up_de.sort_values(["cluster", rank_col])
down_de = down_de.sort_values(["cluster", rank_col])

def top_genes_for_cluster(df, cl, n=20):
    sub = df[df["cluster"].astype(str) == str(cl)].sort_values(rank_col)
    return sub["gene"].astype(str).head(n).tolist()

# panel hit scoring from top100 up genes
rows = []
for cl in cluster_order:
    top20_up = top_genes_for_cluster(up_de, cl, n=20)
    top100_up = top_genes_for_cluster(up_de, cl, n=100)
    top20_down = top_genes_for_cluster(down_de, cl, n=20)
    top100_set = set(top100_up)

    panel_hits = {}
    for panel, genes in MARKER_PANELS.items():
        hits = [g for g in genes if g in top100_set]
        panel_hits[panel] = hits

    best_panels = sorted(panel_hits.items(), key=lambda kv: len(kv[1]), reverse=True)
    best_panel_summary = ";".join([f"{p}:{len(h)}" for p, h in best_panels if len(h) > 0])

    row = {
        "cluster": str(cl),
        "top20_up_genes": ";".join(top20_up),
        "top20_down_genes": ";".join(top20_down),
        "panel_hit_summary_top100_up": best_panel_summary,
    }
    for panel, hits in panel_hits.items():
        row[f"{panel}_n_hits_top100_up"] = len(hits)
        row[f"{panel}_hits_top100_up"] = ";".join(hits)
    rows.append(row)

evidence_auto = pd.DataFrame(rows)

display(evidence_auto[["cluster", "top20_up_genes", "panel_hit_summary_top100_up"]])

,cluster,top20_up_genes,panel_hit_summary_top100_up
0,0,PTPRZ1;BCAN;OLIG1;NOVA1;PTN;ETV1;GPM6A;DBI;MAP...,Tumor_OPC_like:7;Tumor_NPC_like:4;Tumor_AC_like:1
1,1,GPR34;RNASET2;SCIN;C1QC;C3;C1QB;OLR1;TREM2;CSF...,Myeloid_TAM:9;Inflammatory_myeloid:2;Tumor_AC_...
2,2,PBK;TOP2A;BIRC5;UBE2T;NUSAP1;UBE2C;CENPF;TYMS;...,Cycling:8
3,3,TGFBI;SLC16A10;CLEC5A;S100A4;FCGR2B;SLC16A3;TY...,Myeloid_TAM:9;Tumor_MES_like:2;Inflammatory_my...
4,4,CA9;VEGFA;IGFBP5;AKAP12;NRN1;CAV1;HILPDA;LOX;G...,Hypoxia_stress:7;Tumor_MES_like:3;Tumor_AC_lik...
5,5,CD2;CD3D;CD3E;CD3G;TRBC2;TRAC;IL32;GZMA;LCK;CD...,T_NK:7
6,6,MS4A6A;HLA-DPB1;HLA-DRA;HLA-DRB1;HLA-DQB1;HLA-...,Myeloid_TAM:11;Tumor_AC_like:1;Tumor_MES_like:1
7,7,AQP4;GFAP;CLU;ID3;SPARCL1;ID4;KCNN3;CNN3;GJA1;...,Tumor_AC_like:6;Tumor_OPC_like:2
8,8,MAG;KLK6;SPOCK3;CNDP1;ERMN;MOG;NKX6-2;CARNS1;E...,Oligodendrocyte:7
9,9,HBB;GNLY;HBA2;HLA-DPB1;HLA-DPA1;HLA-DRB5;APOE;...,Myeloid_TAM:7;Erythroid_Hb:3;Inflammatory_myel...


In [5]:
# =========================
# Manual annotation map
# =========================

annotation_records = [
    {
        "cluster": "0",
        "annotation": "putative_tumor_OPC_NPC_like",
        "lineage": "putative_tumor_like",
        "malignancy_status": "putative_tumor_like_requires_CNV",
        "confidence": "moderate_high",
        "note": "OPC/NPC-like markers including PTPRZ1, BCAN, OLIG1, PTN. Malignancy requires CNV confirmation.",
    },
    {
        "cluster": "1",
        "annotation": "myeloid_C1Q_TREM2_APOE_TAM",
        "lineage": "myeloid",
        "malignancy_status": "non_malignant_reference_candidate",
        "confidence": "high",
        "note": "C1QC/C1QB/TREM2/CSF1R/APOE-like TAM/macrophage program.",
    },
    {
        "cluster": "2",
        "annotation": "cycling_proliferating_cells",
        "lineage": "cycling_ambiguous",
        "malignancy_status": "requires_marker_context_and_CNV",
        "confidence": "moderate",
        "note": "Strong cell-cycle markers TOP2A/PBK/BIRC5/UBE2C. Underlying lineage requires marker context and CNV.",
    },
    {
        "cluster": "3",
        "annotation": "myeloid_inflammatory_MES_associated",
        "lineage": "myeloid",
        "malignancy_status": "non_malignant_or_MES_associated_requires_CNV_check",
        "confidence": "moderate",
        "note": "TGFBI/S100A4/TYROBP/LYZ suggest inflammatory myeloid/MES-associated macrophage-like cluster; monitor MES ambiguity.",
    },
    {
        "cluster": "4",
        "annotation": "putative_tumor_hypoxia_stress",
        "lineage": "putative_tumor_like_stress",
        "malignancy_status": "putative_tumor_like_requires_CNV",
        "confidence": "moderate_high",
        "note": "CA9/VEGFA/HILPDA/LOX hypoxia stress program; likely tumor-associated but CNV required.",
    },
    {
        "cluster": "5",
        "annotation": "T_NK_cells",
        "lineage": "lymphoid",
        "malignancy_status": "non_malignant_immune",
        "confidence": "high",
        "note": "CD3D/CD3E/TRAC/GZMA/LCK T/NK markers. Check rGBM/DS3 dominance as biological or sampling bias.",
    },
    {
        "cluster": "6",
        "annotation": "myeloid_APC_HLA_CD163_TAM",
        "lineage": "myeloid",
        "malignancy_status": "non_malignant_reference_candidate",
        "confidence": "high",
        "note": "HLA-DRA/HLA-DPB1/CD74/CD163 antigen-presenting macrophage/TAM program.",
    },
    {
        "cluster": "7",
        "annotation": "putative_tumor_AC_like_reactive",
        "lineage": "putative_tumor_like",
        "malignancy_status": "putative_tumor_like_requires_CNV",
        "confidence": "moderate_high",
        "note": "AQP4/GFAP/CLU/SPARCL1 astrocytic/reactive AC-like program; tumor vs normal/reactive requires CNV.",
    },
    {
        "cluster": "8",
        "annotation": "oligodendrocyte_myelinating",
        "lineage": "normal_brain_glial",
        "malignancy_status": "non_malignant_reference_candidate",
        "confidence": "high",
        "note": "MAG/MOG/MOBP/ERMN myelinating oligodendrocyte markers.",
    },
    {
        "cluster": "9",
        "annotation": "ambiguous_mixed_Hb_T_NK_HLA",
        "lineage": "ambiguous_mixed_or_doublet",
        "malignancy_status": "exclude_or_review_before_final_calls",
        "confidence": "low",
        "note": "Mixed HBB/HBA, GNLY, and HLA/myeloid signals; possible mixed/doublet/ambient or condition-specific artifact. Requires manual review.",
    },
    {
        "cluster": "10",
        "annotation": "myeloid_FCN1_inflammatory_monocyte_like",
        "lineage": "myeloid",
        "malignancy_status": "non_malignant_immune",
        "confidence": "moderate_high",
        "note": "FCN1/AQP9/FPR2/LILRA5 inflammatory monocyte-like markers; assess rGBM/DS3 dominance.",
    },
    {
        "cluster": "11",
        "annotation": "putative_tumor_NPC_neuronal_like",
        "lineage": "putative_tumor_like",
        "malignancy_status": "putative_tumor_like_requires_CNV",
        "confidence": "moderate",
        "note": "STMN2/DLX5/DCX/MYT1L neural/NPC-like markers. CNV needed to distinguish tumor-like from neuronal-lineage contamination.",
    },
    {
        "cluster": "12",
        "annotation": "MES_ECM_stromal_like_ambiguous",
        "lineage": "mesenchymal_stromal_ambiguous",
        "malignancy_status": "requires_CNV_and_stromal_marker_review",
        "confidence": "moderate_low",
        "note": "COL1A1/COL3A1/CTHRC1/FABP7 ECM/MES-like signal. Could be stromal/perivascular or MES tumor-like; CNV required.",
    },
    {
        "cluster": "13",
        "annotation": "mural_pericyte_fibroblast_like",
        "lineage": "vascular_stromal",
        "malignancy_status": "non_malignant_stromal_candidate",
        "confidence": "moderate_high",
        "note": "PDGFRB/CD248/LUM/ENPEP pericyte/mural/fibroblast-like markers.",
    },
    {
        "cluster": "14",
        "annotation": "endothelial",
        "lineage": "vascular_endothelial",
        "malignancy_status": "non_malignant_vascular",
        "confidence": "high",
        "note": "KDR/ERG/ESAM/EMCN/CLEC14A endothelial markers.",
    },
    {
        "cluster": "15",
        "annotation": "B_plasma_like",
        "lineage": "lymphoid_B_plasma",
        "malignancy_status": "non_malignant_immune",
        "confidence": "moderate_high",
        "note": "CD79A/POU2AF1/MZB1/IGHM B/plasma-like markers; MS4A1/JCHAIN may be absent from object.",
    },
    {
        "cluster": "16",
        "annotation": "rare_mast_cell_like",
        "lineage": "rare_immune_mast_like",
        "malignancy_status": "non_malignant_immune_review",
        "confidence": "moderate_low",
        "note": "KIT/GATA2/RGS13/ENPP3 mast-like markers; very small cluster, review carefully.",
    },
    {
        "cluster": "17",
        "annotation": "erythroid_Hb_high_or_ambient_RBC",
        "lineage": "erythroid_or_ambient",
        "malignancy_status": "exclude_or_review_before_final_calls",
        "confidence": "moderate",
        "note": "HBA1/HBA2/HBB hemoglobin-high cluster; likely RBC/ambient contamination or rare erythroid cells. Very small cluster.",
    },
]

annotation_table = pd.DataFrame(annotation_records)
annotation_table["cluster"] = annotation_table["cluster"].astype(str)

# Ensure every observed cluster has an annotation row.
missing_ann = sorted(set(cluster_order) - set(annotation_table["cluster"]), key=lambda x: int(x) if str(x).isdigit() else str(x))
if missing_ann:
    raise ValueError(f"Missing annotation records for clusters: {missing_ann}")

extra_ann = sorted(set(annotation_table["cluster"]) - set(cluster_order))
if extra_ann:
    print("[WARNING] annotation table contains clusters not observed:", extra_ann)

annotation_table = annotation_table.sort_values("cluster", key=lambda s: s.astype(str).map(lambda x: int(x) if x.isdigit() else 999999))
display(annotation_table)


if annotation_table.shape != EXPECTED_ANNOTATION_TABLE_SHAPE:
    raise AssertionError(
        "Manual annotation table shape changed: "
        f"{annotation_table.shape}"
    )

,cluster,annotation,lineage,malignancy_status,confidence,note
0,0,putative_tumor_OPC_NPC_like,putative_tumor_like,putative_tumor_like_requires_CNV,moderate_high,"OPC/NPC-like markers including PTPRZ1, BCAN, O..."
1,1,myeloid_C1Q_TREM2_APOE_TAM,myeloid,non_malignant_reference_candidate,high,C1QC/C1QB/TREM2/CSF1R/APOE-like TAM/macrophage...
2,2,cycling_proliferating_cells,cycling_ambiguous,requires_marker_context_and_CNV,moderate,Strong cell-cycle markers TOP2A/PBK/BIRC5/UBE2...
3,3,myeloid_inflammatory_MES_associated,myeloid,non_malignant_or_MES_associated_requires_CNV_c...,moderate,TGFBI/S100A4/TYROBP/LYZ suggest inflammatory m...
4,4,putative_tumor_hypoxia_stress,putative_tumor_like_stress,putative_tumor_like_requires_CNV,moderate_high,CA9/VEGFA/HILPDA/LOX hypoxia stress program; l...
5,5,T_NK_cells,lymphoid,non_malignant_immune,high,CD3D/CD3E/TRAC/GZMA/LCK T/NK markers. Check rG...
6,6,myeloid_APC_HLA_CD163_TAM,myeloid,non_malignant_reference_candidate,high,HLA-DRA/HLA-DPB1/CD74/CD163 antigen-presenting...
7,7,putative_tumor_AC_like_reactive,putative_tumor_like,putative_tumor_like_requires_CNV,moderate_high,AQP4/GFAP/CLU/SPARCL1 astrocytic/reactive AC-l...
8,8,oligodendrocyte_myelinating,normal_brain_glial,non_malignant_reference_candidate,high,MAG/MOG/MOBP/ERMN myelinating oligodendrocyte ...
9,9,ambiguous_mixed_Hb_T_NK_HLA,ambiguous_mixed_or_doublet,exclude_or_review_before_final_calls,low,"Mixed HBB/HBA, GNLY, and HLA/myeloid signals; ..."


In [6]:
# =========================
# Compute bias / composition flags from cluster QC
# =========================

if cluster_qc.empty:
    print("[INFO] cluster_qc file missing; computing basic QC from obs")
    rows = []
    for cl in cluster_order:
        obs_sub = obs[obs[CLUSTER_KEY].astype(str) == str(cl)]
        row = {"cluster": str(cl), "n_cells": int(obs_sub.shape[0])}
        for col in ["core_dataset", "condition", "patient_id", "sample_id"]:
            if col in obs_sub.columns:
                vc = obs_sub[col].astype(str).value_counts(dropna=False)
                row[f"top_{col}"] = vc.index[0] if len(vc) else "NA"
                row[f"top_{col}_fraction"] = float(vc.iloc[0] / obs_sub.shape[0]) if len(vc) and obs_sub.shape[0] > 0 else np.nan
                row[f"n_{col}"] = int(vc.shape[0])
        rows.append(row)
    cluster_qc_use = pd.DataFrame(rows)
else:
    cluster_qc_use = cluster_qc.copy()

# Normalize column names from existing QC table.
rename_map = {
    "top_core_dataset": "top_core_dataset",
    "top_core_dataset_fraction": "top_core_dataset_fraction",
    "top_condition": "top_condition",
    "top_condition_fraction": "top_condition_fraction",
    "top_patient_id": "top_patient_id",
    "top_patient_id_fraction": "top_patient_id_fraction",
    "top_sample_id": "top_sample_id",
    "top_sample_id_fraction": "top_sample_id_fraction",
}
cluster_qc_use = cluster_qc_use.rename(columns=rename_map)
cluster_qc_use["cluster"] = cluster_qc_use["cluster"].astype(str)

# Create standard flags.
def bool_col(df, col, default=False):
    return df[col].astype(bool) if col in df.columns else pd.Series(default, index=df.index)

bias = cluster_qc_use.copy()
if "n_cells" not in bias.columns:
    bias["n_cells"] = bias["cluster"].map(obs[CLUSTER_KEY].astype(str).value_counts().to_dict()).astype(int)

bias["flag_small_lt_500"] = bias["n_cells"].astype(int) < 500
bias["flag_small_lt_100"] = bias["n_cells"].astype(int) < 100
bias["flag_dataset_dominant_gt_0_80"] = bias.get("top_core_dataset_fraction", pd.Series(0, index=bias.index)).astype(float) > 0.80
bias["flag_condition_dominant_gt_0_80"] = bias.get("top_condition_fraction", pd.Series(0, index=bias.index)).astype(float) > 0.80
bias["flag_patient_dominant_gt_0_80"] = bias.get("top_patient_id_fraction", pd.Series(0, index=bias.index)).astype(float) > 0.80
bias["flag_sample_dominant_gt_0_80"] = bias.get("top_sample_id_fraction", pd.Series(0, index=bias.index)).astype(float) > 0.80

flag_cols = [c for c in bias.columns if c.startswith("flag_")]
bias["n_bias_or_size_flags"] = bias[flag_cols].sum(axis=1).astype(int)
bias["bias_flag_summary"] = bias[flag_cols].apply(lambda r: ";".join([c for c, v in r.items() if bool(v)]) if any(r) else "none", axis=1)

write_tsv_replace(bias, ANNOTATION_BIAS_TSV)
display(bias[["cluster", "n_cells", "bias_flag_summary"] + [c for c in ["top_core_dataset", "top_core_dataset_fraction", "top_condition", "top_condition_fraction", "top_patient_id", "top_patient_id_fraction", "top_sample_id", "top_sample_id_fraction"] if c in bias.columns]])


if bias.shape != EXPECTED_BIAS_SHAPE:
    raise AssertionError(
        "Bias table shape changed: "
        f"{bias.shape}"
    )

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\annotation_marker_based_HVG8000_r0_6_bias_flags.tsv shape=(18, 29)


,cluster,n_cells,bias_flag_summary,top_core_dataset,top_core_dataset_fraction,top_condition,top_condition_fraction,top_patient_id,top_patient_id_fraction,top_sample_id,top_sample_id_fraction
0,0,61646,none,DS3_GSE182109,0.555900,GBM,0.444100,04,0.185527,GSM5518602_ndGBM-01-D,0.064692
1,1,59791,none,DS3_GSE182109,0.669649,ndGBM,0.426586,01,0.148534,GSM5518603_ndGBM-01-F,0.062953
2,2,38634,none,DS3_GSE182109,0.552674,GBM,0.447326,04,0.240358,GSM5518621_rGBM-04-3,0.057203
3,3,35355,none,DS3_GSE182109,0.746599,rGBM,0.586225,01,0.470117,GSM5518597_rGBM-01-B,0.217282
4,4,32896,none,DS4_GSE278450,0.466652,GBM,0.606548,01,0.151751,GSM8546833_BT356,0.149471
5,5,23977,flag_dataset_dominant_gt_0_80,DS3_GSE182109,0.967052,rGBM,0.617091,02,0.555824,GSM5518615_rGBM-02-5,0.194603
6,6,22943,none,DS3_GSE182109,0.633483,GBM,0.366517,02,0.157259,GSM5518636_ndGBM-07,0.089134
7,7,22525,none,DS3_GSE182109,0.385838,GBM,0.614162,04,0.121154,GSM8546834_BT363,0.120844
8,8,14809,none,DS4_GSE278450,0.558714,GBM,0.610777,04,0.179485,GSM8546845_BT396,0.113445
9,9,13864,flag_dataset_dominant_gt_0_80;flag_condition_d...,DS3_GSE182109,0.941864,ndGBM,0.806621,10,0.696336,GSM5518629_ndGBM-10,0.696336


In [7]:
# =========================
# Build final annotation evidence table
# =========================

ann = annotation_table.merge(evidence_auto, on="cluster", how="left")
ann = ann.merge(bias[["cluster", "n_cells", "bias_flag_summary", "n_bias_or_size_flags"] + [c for c in bias.columns if c.startswith("top_")]], on="cluster", how="left")

# Caution rules combine manual notes and QC flags.
def assign_caution(row):
    caution = []
    if str(row.get("confidence", "")).lower() in {"low", "moderate_low"}:
        caution.append("low_or_moderate_low_confidence")
    if str(row.get("malignancy_status", "")).startswith("putative_tumor") or "requires_CNV" in str(row.get("malignancy_status", "")):
        caution.append("requires_CNV")
    if row.get("bias_flag_summary", "none") != "none":
        caution.append(str(row.get("bias_flag_summary")))
    if "ambiguous" in str(row.get("lineage", "")) or "ambiguous" in str(row.get("annotation", "")):
        caution.append("ambiguous_annotation")
    if "exclude_or_review" in str(row.get("malignancy_status", "")):
        caution.append("review_or_exclude")
    return ";".join(dict.fromkeys([c for c in caution if c])) if caution else "none"

ann["annotation_caution"] = ann.apply(assign_caution, axis=1)

write_tsv_replace(annotation_table, ANNOTATION_TSV)
write_tsv_replace(ann, ANNOTATION_EVIDENCE_TSV)

display(ann[["cluster", "annotation", "lineage", "malignancy_status", "confidence", "n_cells", "bias_flag_summary", "annotation_caution", "top20_up_genes"]])


if annotation_table.shape != EXPECTED_ANNOTATION_TABLE_SHAPE:
    raise AssertionError(
        "Annotation table shape changed."
    )

if ann.shape != EXPECTED_EVIDENCE_SHAPE:
    raise AssertionError(
        "Annotation evidence shape changed: "
        f"{ann.shape}"
    )

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\annotation_marker_based_HVG8000_r0_6.tsv shape=(18, 6)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\cluster_marker_annotation_evidence_HVG8000_r0_6.tsv shape=(18, 53)


,cluster,annotation,lineage,malignancy_status,confidence,n_cells,bias_flag_summary,annotation_caution,top20_up_genes
0,0,putative_tumor_OPC_NPC_like,putative_tumor_like,putative_tumor_like_requires_CNV,moderate_high,61646,none,requires_CNV,PTPRZ1;BCAN;OLIG1;NOVA1;PTN;ETV1;GPM6A;DBI;MAP...
1,1,myeloid_C1Q_TREM2_APOE_TAM,myeloid,non_malignant_reference_candidate,high,59791,none,none,GPR34;RNASET2;SCIN;C1QC;C3;C1QB;OLR1;TREM2;CSF...
2,2,cycling_proliferating_cells,cycling_ambiguous,requires_marker_context_and_CNV,moderate,38634,none,ambiguous_annotation,PBK;TOP2A;BIRC5;UBE2T;NUSAP1;UBE2C;CENPF;TYMS;...
3,3,myeloid_inflammatory_MES_associated,myeloid,non_malignant_or_MES_associated_requires_CNV_c...,moderate,35355,none,requires_CNV,TGFBI;SLC16A10;CLEC5A;S100A4;FCGR2B;SLC16A3;TY...
4,4,putative_tumor_hypoxia_stress,putative_tumor_like_stress,putative_tumor_like_requires_CNV,moderate_high,32896,none,requires_CNV,CA9;VEGFA;IGFBP5;AKAP12;NRN1;CAV1;HILPDA;LOX;G...
5,5,T_NK_cells,lymphoid,non_malignant_immune,high,23977,flag_dataset_dominant_gt_0_80,flag_dataset_dominant_gt_0_80,CD2;CD3D;CD3E;CD3G;TRBC2;TRAC;IL32;GZMA;LCK;CD...
6,6,myeloid_APC_HLA_CD163_TAM,myeloid,non_malignant_reference_candidate,high,22943,none,none,MS4A6A;HLA-DPB1;HLA-DRA;HLA-DRB1;HLA-DQB1;HLA-...
7,7,putative_tumor_AC_like_reactive,putative_tumor_like,putative_tumor_like_requires_CNV,moderate_high,22525,none,requires_CNV,AQP4;GFAP;CLU;ID3;SPARCL1;ID4;KCNN3;CNN3;GJA1;...
8,8,oligodendrocyte_myelinating,normal_brain_glial,non_malignant_reference_candidate,high,14809,none,none,MAG;KLK6;SPOCK3;CNDP1;ERMN;MOG;NKX6-2;CARNS1;E...
9,9,ambiguous_mixed_Hb_T_NK_HLA,ambiguous_mixed_or_doublet,exclude_or_review_before_final_calls,low,13864,flag_dataset_dominant_gt_0_80;flag_condition_d...,low_or_moderate_low_confidence;flag_dataset_do...,HBB;GNLY;HBA2;HLA-DPB1;HLA-DPA1;HLA-DRB5;APOE;...


In [8]:
# =========================
# 7) APPLY ANNOTATION TO METADATA + HISTORICAL COUNT TABLES
# =========================

annotation_map = dict(
    zip(
        annotation_table[
            "cluster"
        ],
        annotation_table[
            "annotation"
        ],
    )
)

lineage_map = dict(
    zip(
        annotation_table[
            "cluster"
        ],
        annotation_table[
            "lineage"
        ],
    )
)

confidence_map = dict(
    zip(
        annotation_table[
            "cluster"
        ],
        annotation_table[
            "confidence"
        ],
    )
)

malignancy_map = dict(
    zip(
        annotation_table[
            "cluster"
        ],
        annotation_table[
            "malignancy_status"
        ],
    )
)

note_map = dict(
    zip(
        annotation_table[
            "cluster"
        ],
        annotation_table[
            "note"
        ],
    )
)

caution_map = dict(
    zip(
        ann[
            "cluster"
        ],
        ann[
            "annotation_caution"
        ],
    )
)


annotated_obs = (
    obs.copy()
)

cluster_series = (
    annotated_obs[
        CLUSTER_KEY
    ]
    .astype(str)
)


annotated_obs[
    ANNOTATION_COL
] = (
    cluster_series
    .map(
        annotation_map
    )
    .fillna(
        "unannotated"
    )
    .astype(
        "category"
    )
)

annotated_obs[
    LINEAGE_COL
] = (
    cluster_series
    .map(
        lineage_map
    )
    .fillna(
        "unannotated"
    )
    .astype(
        "category"
    )
)

annotated_obs[
    CONFIDENCE_COL
] = (
    cluster_series
    .map(
        confidence_map
    )
    .fillna(
        "unannotated"
    )
    .astype(
        "category"
    )
)

annotated_obs[
    MALIGNANCY_COL
] = (
    cluster_series
    .map(
        malignancy_map
    )
    .fillna(
        "unannotated"
    )
    .astype(
        "category"
    )
)

annotated_obs[
    CAUTION_COL
] = (
    cluster_series
    .map(
        caution_map
    )
    .fillna(
        "none"
    )
    .astype(
        "category"
    )
)

annotated_obs[
    NOTE_COL
] = (
    cluster_series
    .map(
        note_map
    )
    .fillna(
        ""
    )
    .astype(str)
)


counts = (
    annotated_obs
    .groupby(
        [
            CLUSTER_KEY,
            ANNOTATION_COL,
            LINEAGE_COL,
            CONFIDENCE_COL,
            MALIGNANCY_COL,
            CAUTION_COL,
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="n_cells"
    )
    .sort_values(
        CLUSTER_KEY,
        key=lambda series:
            series
            .astype(str)
            .map(
                lambda value:
                    int(value)
                    if value.isdigit()
                    else 999999
            ),
    )
)


counts[
    "fraction_total"
] = (
    counts[
        "n_cells"
    ]
    / EXPECTED_N_CELLS
)


write_tsv_replace(
    counts,
    ANNOTATION_COUNTS_TSV,
)

display(
    counts
)


if counts.shape != EXPECTED_COUNTS_SHAPE:
    raise AssertionError(
        "Annotation cell-count table shape changed: "
        f"{counts.shape}"
    )


if not cluster_comp.empty:
    comp_annot = (
        cluster_comp
        .merge(
            annotation_table,
            on="cluster",
            how="left",
        )
    )

    write_tsv_replace(
        comp_annot,
        ANNOTATION_COMPOSITION_TSV,
    )

    display(
        comp_annot.head(
            20
        )
    )

    if (
        comp_annot.shape
        != EXPECTED_ANNOTATED_COMPOSITION_SHAPE
    ):
        raise AssertionError(
            "Annotated composition shape changed: "
            f"{comp_annot.shape}"
        )

else:
    raise RuntimeError(
        "Historical cluster composition table is required."
    )


if set(
    counts[
        ANNOTATION_COL
    ].astype(str)
) != set(
    annotation_table[
        "annotation"
    ].astype(str)
):
    raise AssertionError(
        "Final annotation labels do not match "
        "the 18-cluster annotation table."
    )


print(
    "\n[HISTORICAL 18-CLUSTER ANNOTATION MAP APPLIED]"
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\annotation_marker_based_HVG8000_r0_6_cell_counts.tsv shape=(18, 8)


,leiden_scvi_main,annotation_marker_based_HVG8000_r0_6,lineage_marker_based_HVG8000_r0_6,annotation_confidence_HVG8000_r0_6,malignancy_status_marker_based_HVG8000_r0_6,annotation_caution_HVG8000_r0_6,n_cells,fraction_total
0,0,putative_tumor_OPC_NPC_like,putative_tumor_like,moderate_high,putative_tumor_like_requires_CNV,requires_CNV,61646,0.175416
1,1,myeloid_C1Q_TREM2_APOE_TAM,myeloid,high,non_malignant_reference_candidate,none,59791,0.170138
10,2,cycling_proliferating_cells,cycling_ambiguous,moderate,requires_marker_context_and_CNV,ambiguous_annotation,38634,0.109935
11,3,myeloid_inflammatory_MES_associated,myeloid,moderate,non_malignant_or_MES_associated_requires_CNV_c...,requires_CNV,35355,0.100604
12,4,putative_tumor_hypoxia_stress,putative_tumor_like_stress,moderate_high,putative_tumor_like_requires_CNV,requires_CNV,32896,0.093607
13,5,T_NK_cells,lymphoid,high,non_malignant_immune,flag_dataset_dominant_gt_0_80,23977,0.068228
14,6,myeloid_APC_HLA_CD163_TAM,myeloid,high,non_malignant_reference_candidate,none,22943,0.065285
15,7,putative_tumor_AC_like_reactive,putative_tumor_like,moderate_high,putative_tumor_like_requires_CNV,requires_CNV,22525,0.064096
16,8,oligodendrocyte_myelinating,normal_brain_glial,high,non_malignant_reference_candidate,none,14809,0.042140
17,9,ambiguous_mixed_Hb_T_NK_HLA,ambiguous_mixed_or_doublet,low,exclude_or_review_before_final_calls,low_or_moderate_low_confidence;flag_dataset_do...,13864,0.039451


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\annotation_marker_based_HVG8000_r0_6_composition_annotated.tsv shape=(2538, 10)


,cluster,metadata_column,metadata_value,n_cells,fraction_within_cluster,annotation,lineage,malignancy_status,confidence,note
0,0,core_dataset,DS1_GSE103224,6607,0.107176,putative_tumor_OPC_NPC_like,putative_tumor_like,putative_tumor_like_requires_CNV,moderate_high,"OPC/NPC-like markers including PTPRZ1, BCAN, O..."
1,0,core_dataset,DS2_GSE141383,7202,0.116828,putative_tumor_OPC_NPC_like,putative_tumor_like,putative_tumor_like_requires_CNV,moderate_high,"OPC/NPC-like markers including PTPRZ1, BCAN, O..."
2,0,core_dataset,DS3_GSE182109,34269,0.555900,putative_tumor_OPC_NPC_like,putative_tumor_like,putative_tumor_like_requires_CNV,moderate_high,"OPC/NPC-like markers including PTPRZ1, BCAN, O..."
3,0,core_dataset,DS4_GSE278450,13568,0.220095,putative_tumor_OPC_NPC_like,putative_tumor_like,putative_tumor_like_requires_CNV,moderate_high,"OPC/NPC-like markers including PTPRZ1, BCAN, O..."
4,1,core_dataset,DS1_GSE103224,1538,0.025723,myeloid_C1Q_TREM2_APOE_TAM,myeloid,non_malignant_reference_candidate,high,C1QC/C1QB/TREM2/CSF1R/APOE-like TAM/macrophage...
5,1,core_dataset,DS2_GSE141383,808,0.013514,myeloid_C1Q_TREM2_APOE_TAM,myeloid,non_malignant_reference_candidate,high,C1QC/C1QB/TREM2/CSF1R/APOE-like TAM/macrophage...
6,1,core_dataset,DS3_GSE182109,40039,0.669649,myeloid_C1Q_TREM2_APOE_TAM,myeloid,non_malignant_reference_candidate,high,C1QC/C1QB/TREM2/CSF1R/APOE-like TAM/macrophage...
7,1,core_dataset,DS4_GSE278450,17406,0.291114,myeloid_C1Q_TREM2_APOE_TAM,myeloid,non_malignant_reference_candidate,high,C1QC/C1QB/TREM2/CSF1R/APOE-like TAM/macrophage...
8,10,core_dataset,DS1_GSE103224,1,0.000158,myeloid_FCN1_inflammatory_monocyte_like,myeloid,non_malignant_immune,moderate_high,FCN1/AQP9/FPR2/LILRA5 inflammatory monocyte-li...
9,10,core_dataset,DS2_GSE141383,1,0.000158,myeloid_FCN1_inflammatory_monocyte_like,myeloid,non_malignant_immune,moderate_high,FCN1/AQP9/FPR2/LILRA5 inflammatory monocyte-li...



[HISTORICAL 18-CLUSTER ANNOTATION MAP APPLIED]


In [9]:
# =========================
# 8) DECISION REPORT
# =========================

decision_rows = [
    {
        "item":
            "input_h5ad",
        "value":
            str(
                FULLGENE_INPUT_H5AD
            ),
    },
    {
        "item":
            "output_h5ad",
        "value":
            str(
                ANNOTATED_H5AD
            ),
    },
    {
        "item":
            "output_h5ad_alias",
        "value":
            str(
                ANNOTATED_H5AD_ALIAS
            ),
    },
    {
        "item":
            "cluster_key_main",
        "value":
            CLUSTER_KEY,
    },
    {
        "item":
            "source_cluster_key",
        "value":
            SOURCE_CLUSTER_KEY,
    },
    {
        "item":
            "n_cells",
        "value":
            str(
                EXPECTED_N_CELLS
            ),
    },
    {
        "item":
            "n_genes",
        "value":
            str(
                EXPECTED_N_GENES
            ),
    },
    {
        "item":
            "n_clusters",
        "value":
            str(
                len(
                    cluster_order
                )
            ),
    },
    {
        "item":
            "annotation_col",
        "value":
            ANNOTATION_COL,
    },
    {
        "item":
            "lineage_col",
        "value":
            LINEAGE_COL,
    },
    {
        "item":
            "confidence_col",
        "value":
            CONFIDENCE_COL,
    },
    {
        "item":
            "malignancy_col",
        "value":
            MALIGNANCY_COL,
    },
    {
        "item":
            "caution_col",
        "value":
            CAUTION_COL,
    },
    {
        "item":
            "principle",
        "value":
            (
                "Tumor-like labels are putative "
                "until CNV validation."
            ),
    },
    {
        "item":
            "main_recommendation",
        "value":
            (
                "Proceed to marker review / annotation "
                "refinement, then CNV-based malignant validation."
            ),
    },
]


decision_report = pd.DataFrame(
    decision_rows
)


write_tsv_replace(
    decision_report,
    ANNOTATION_DECISION_REPORT_TSV,
)

display(
    decision_report
)


if (
    decision_report.shape
    != EXPECTED_DECISION_REPORT_SHAPE
):
    raise AssertionError(
        "Decision-report shape changed: "
        f"{decision_report.shape}"
    )

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\annotation_marker_based_HVG8000_r0_6_decision_report.tsv shape=(15, 2)


,item,value
0,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...
1,output_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...
2,output_h5ad_alias,G:\Repository\glioma-cnv-single-cell-atlas\dat...
3,cluster_key_main,leiden_scvi_main
4,source_cluster_key,leiden_scvi_HVG8000_r0_6
5,n_cells,351427
6,n_genes,15176
7,n_clusters,18
8,annotation_col,annotation_marker_based_HVG8000_r0_6
9,lineage_col,lineage_marker_based_HVG8000_r0_6


## Visual checks

The historical UMAP panels use only metadata plus `X_umap`, so they are cheap
and exact.

The two marker dotplots historically normalized a **59-gene raw-count subset**
to a per-cell target sum of `1e4`. To preserve that exact plotting semantics
without loading the full expression matrix, the next cell scans the CSR counts
layer sequentially and extracts only those 59 genes.

In [10]:
# =========================
# 9) UMAP PLOTS — LIGHTWEIGHT
# =========================

umap_adata = ad.AnnData(
    X=None,
    obs=annotated_obs.copy(),
)

if x_umap is not None:
    umap_adata.obsm[
        "X_umap"
    ] = x_umap


if "X_umap" not in umap_adata.obsm:
    print(
        "[WARNING] X_umap not found; "
        "skipping UMAP plots"
    )

else:
    for color_col in [
        ANNOTATION_COL,
        LINEAGE_COL,
        CONFIDENCE_COL,
        MALIGNANCY_COL,
        CAUTION_COL,
        CLUSTER_KEY,
        "core_dataset",
        "condition",
    ]:
        if color_col not in (
            umap_adata.obs.columns
        ):
            continue

        sc.pl.umap(
            umap_adata,
            color=color_col,
            legend_loc="right margin",
            frameon=False,
            size=3,
            show=False,
        )

        figure = plt.gcf()

        output = (
            FIG_DIR
            / f"umap_{color_col}.png"
        )

        figure.savefig(
            output,
            dpi=180,
            bbox_inches="tight",
        )

        plt.close(
            figure
        )

        print(
            f"[SAVED] {output}"
        )

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\annotation_marker_based_HVG8000_r0_6\umap_annotation_marker_based_HVG8000_r0_6.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\annotation_marker_based_HVG8000_r0_6\umap_lineage_marker_based_HVG8000_r0_6.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\annotation_marker_based_HVG8000_r0_6\umap_annotation_confidence_HVG8000_r0_6.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\annotation_marker_based_HVG8000_r0_6\umap_malignancy_status_marker_based_HVG8000_r0_6.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\annotation_marker_based_HVG8000_r0_6\umap_annotation_caution_HVG8000_r0_6.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\annotation_marker_based_HVG8000_r0_6\umap_leiden_scvi_main.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\annotation_marker_based_HVG8000_r0_6\umap_core_dataset.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\fi

In [11]:
# =========================
# 10) HISTORICAL 59-GENE DOTPLOT — MEMORY SAFE
# =========================

plot_marker_panels = {
    "Tumor_OPC_NPC": [
        "PTPRZ1", "BCAN", "OLIG1", "OLIG2", "PDGFRA",
        "PTN", "NES", "SOX4", "STMN2", "DCX",
    ],
    "Tumor_AC_MES_Hypoxia": [
        "AQP4", "GFAP", "CLU", "SPARCL1", "CD44",
        "TGFBI", "CHI3L1", "CA9", "VEGFA", "HILPDA",
    ],
    "Myeloid": [
        "LYZ", "TYROBP", "C1QA", "C1QB", "C1QC",
        "TREM2", "CSF1R", "CD74", "HLA-DRA", "FCN1",
    ],
    "Lymphoid": [
        "CD3D", "CD3E", "TRAC", "LCK", "NKG7",
        "GNLY", "GZMB", "CD79A", "MZB1", "IGHM",
    ],
    "Vascular_Stromal": [
        "PECAM1", "VWF", "KDR", "ESAM", "PDGFRB",
        "ACTA2", "RGS5", "COL1A1", "COL3A1", "LUM",
    ],
    "Other_rare": [
        "MOG", "MAG", "MOBP", "HBA1", "HBA2",
        "HBB", "KIT", "GATA2", "RGS13",
    ],
}


flat_genes = []

for genes in (
    plot_marker_panels.values()
):
    flat_genes.extend(
        genes
    )


flat_genes = list(
    dict.fromkeys(
        flat_genes
    )
)


present = present_genes(
    flat_genes,
    var_names,
)

missing = [
    gene
    for gene
    in flat_genes
    if gene not in present
]


print(
    "plot marker genes present:",
    len(
        present
    ),
)

print(
    "plot marker genes missing:",
    missing,
)


if len(
    present
) != EXPECTED_PLOT_MARKER_GENES:
    raise AssertionError(
        "Historical dotplot marker-gene count changed: "
        f"{len(present)}"
    )

if missing:
    raise AssertionError(
        "Historical dotplot unexpectedly lost genes: "
        f"{missing}"
    )


plot_panels_present = {
    panel:
        present_genes(
            genes,
            var_names,
        )
    for panel, genes
    in plot_marker_panels.items()
}

plot_panels_present = {
    panel:
        genes
    for panel, genes
    in plot_panels_present.items()
    if len(
        genes
    ) > 0
}


def read_counts_gene_subset(
    h5ad_path,
    selected_genes,
):
    """
    Sequentially scan the CSR raw-count layer and return only selected columns.
    This preserves the historical plot semantics while avoiding full-matrix RAM.
    """

    selected_genes = list(
        selected_genes
    )

    global_columns = np.asarray(
        [
            int(
                var_names.get_loc(
                    gene
                )
            )
            for gene
            in selected_genes
        ],
        dtype=np.int32,
    )


    subset_blocks = []


    with h5py.File(
        h5ad_path,
        "r",
    ) as handle:

        counts = (
            handle[
                "layers"
            ][
                "counts"
            ]
        )

        indptr = np.asarray(
            counts[
                "indptr"
            ],
            dtype=np.int64,
        )

        data_ds = counts[
            "data"
        ]

        indices_ds = counts[
            "indices"
        ]

        total_nnz = int(
            data_ds.shape[
                0
            ]
        )


        r0 = 0

        while r0 < EXPECTED_N_CELLS:

            d0 = int(
                indptr[
                    r0
                ]
            )

            target_d1 = min(
                d0
                + MAX_NNZ_PER_ROW_BLOCK,
                total_nnz,
            )

            r1 = int(
                np.searchsorted(
                    indptr,
                    target_d1,
                    side="right",
                )
                - 1
            )

            r1 = max(
                r1,
                r0 + 1,
            )

            r1 = min(
                r1,
                EXPECTED_N_CELLS,
            )

            d1 = int(
                indptr[
                    r1
                ]
            )


            block_data = np.asarray(
                data_ds[
                    d0:d1
                ]
            )

            block_indices = np.asarray(
                indices_ds[
                    d0:d1
                ],
                dtype=np.int32,
            )

            block_indptr = (
                indptr[
                    r0:r1 + 1
                ]
                - d0
            ).astype(
                np.int64,
                copy=True,
            )


            block = sp.csr_matrix(
                (
                    block_data,
                    block_indices,
                    block_indptr,
                ),
                shape=(
                    r1 - r0,
                    EXPECTED_N_GENES,
                ),
            )


            subset = (
                block[
                    :,
                    global_columns,
                ]
                .tocsr()
            )

            subset_blocks.append(
                subset
            )


            print(
                "[DOTPLOT COUNTS]",
                f"rows {r0:,}-{r1:,}",
                f"selected nnz={subset.nnz:,}",
            )


            del block
            del block_data
            del block_indices
            del block_indptr

            gc.collect()

            r0 = r1


    matrix = sp.vstack(
        subset_blocks,
        format="csr",
    )

    del subset_blocks
    gc.collect()


    if matrix.shape != (
        EXPECTED_N_CELLS,
        len(
            selected_genes
        ),
    ):
        raise AssertionError(
            "Marker-subset matrix shape changed: "
            f"{matrix.shape}"
        )


    return matrix


plot_counts = read_counts_gene_subset(
    FULLGENE_INPUT_H5AD,
    present,
)


plot_adata = ad.AnnData(
    X=plot_counts,
    obs=annotated_obs.copy(),
    var=pd.DataFrame(
        index=pd.Index(
            present,
            dtype=str,
        ),
    ),
)


# EXACT historical plotting semantics:
# normalize only this 59-gene raw-count plotting subset, then log1p.
sc.pp.normalize_total(
    plot_adata,
    target_sum=1e4,
)

sc.pp.log1p(
    plot_adata
)


sc.pl.dotplot(
    plot_adata,
    var_names=plot_panels_present,
    groupby=ANNOTATION_COL,
    standard_scale="var",
    dendrogram=False,
    show=False,
)

figure = plt.gcf()

output = (
    FIG_DIR
    / "dotplot_marker_panels_by_annotation.png"
)

figure.savefig(
    output,
    dpi=180,
    bbox_inches="tight",
)

plt.close(
    figure
)

print(
    f"[SAVED] {output}"
)


sc.pl.dotplot(
    plot_adata,
    var_names=plot_panels_present,
    groupby=CLUSTER_KEY,
    standard_scale="var",
    dendrogram=False,
    show=False,
)

figure = plt.gcf()

output = (
    FIG_DIR
    / "dotplot_marker_panels_by_cluster.png"
)

figure.savefig(
    output,
    dpi=180,
    bbox_inches="tight",
)

plt.close(
    figure
)

print(
    f"[SAVED] {output}"
)


del plot_adata
del plot_counts
gc.collect()

plot marker genes present: 59
plot marker genes missing: []
[DOTPLOT COUNTS] rows 0-4,241 selected nnz=34,438
[DOTPLOT COUNTS] rows 4,241-11,672 selected nnz=51,772
[DOTPLOT COUNTS] rows 11,672-21,842 selected nnz=66,962
[DOTPLOT COUNTS] rows 21,842-31,237 selected nnz=60,121
[DOTPLOT COUNTS] rows 31,237-39,049 selected nnz=57,917
[DOTPLOT COUNTS] rows 39,049-45,782 selected nnz=55,645
[DOTPLOT COUNTS] rows 45,782-49,725 selected nnz=44,252
[DOTPLOT COUNTS] rows 49,725-54,177 selected nnz=50,492
[DOTPLOT COUNTS] rows 54,177-58,699 selected nnz=51,059
[DOTPLOT COUNTS] rows 58,699-63,156 selected nnz=49,559
[DOTPLOT COUNTS] rows 63,156-67,223 selected nnz=47,434
[DOTPLOT COUNTS] rows 67,223-70,841 selected nnz=45,097
[DOTPLOT COUNTS] rows 70,841-74,149 selected nnz=38,851
[DOTPLOT COUNTS] rows 74,149-77,428 selected nnz=37,638
[DOTPLOT COUNTS] rows 77,428-80,846 selected nnz=38,335
[DOTPLOT COUNTS] rows 80,846-84,360 selected nnz=39,047
[DOTPLOT COUNTS] rows 84,360-88,259 selected nnz=43

12

In [12]:
# =========================
# 11) COPY CANONICAL BACKBONE + PATCH ANNOTATION
# =========================

if ANNOTATED_BUILDING_H5AD.exists():
    ANNOTATED_BUILDING_H5AD.unlink()


input_size = int(
    FULLGENE_INPUT_H5AD
    .stat()
    .st_size
)

free_space = int(
    shutil.disk_usage(
        ANNOTATED_H5AD.parent
    ).free
)


if free_space < (
    input_size
    + 5 * 1024**3
):
    raise OSError(
        "Not enough free space to publish "
        "the annotated H5AD safely."
    )


print(
    "[COPY] canonical full-gene backbone"
)

print(
    FULLGENE_INPUT_H5AD
)

print(
    "->"
)

print(
    ANNOTATED_BUILDING_H5AD
)


shutil.copyfile(
    FULLGENE_INPUT_H5AD,
    ANNOTATED_BUILDING_H5AD,
)


print(
    "[COPY COMPLETE]"
)


annotation_provenance = {
    "cluster_key":
        CLUSTER_KEY,

    "source_cluster_key":
        SOURCE_CLUSTER_KEY,

    "annotation_col":
        ANNOTATION_COL,

    "lineage_col":
        LINEAGE_COL,

    "confidence_col":
        CONFIDENCE_COL,

    "malignancy_col":
        MALIGNANCY_COL,

    "caution_col":
        CAUTION_COL,

    "note_col":
        NOTE_COL,

    "principle":
        (
            "Tumor-like labels are putative "
            "until CNV validation."
        ),

    "canonical_historical_18_cluster_branch":
        True,

    "marker_de_historical_checkpoints_reproduced_in_notebook_07":
        True,

    "full_expression_matrix_loaded_into_ram":
        False,

    "final_project_dataset_nomenclature":
        "DS1/DS2/DS3/DS4",
}


with h5py.File(
    ANNOTATED_BUILDING_H5AD,
    "r+",
) as handle:

    replace_h5_key(
        handle,
        "obs",
        annotated_obs,
    )


    if "uns" not in handle:
        write_elem(
            handle,
            "uns",
            {},
        )


    # Historical notebook removed stale rank_genes_groups objects before save.
    old_rank_keys = [
        key
        for key
        in list(
            handle[
                "uns"
            ].keys()
        )
        if str(
            key
        ).startswith(
            "rank_genes_groups"
        )
    ]

    for key in old_rank_keys:
        del handle[
            "uns"
        ][
            key
        ]

        print(
            "[REMOVED old uns key]",
            key,
        )


    replace_h5_key(
        handle[
            "uns"
        ],
        "marker_based_annotation_HVG8000_r0_6",
        annotation_provenance,
    )


    handle.flush()


print(
    "[ANNOTATION PATCH COMPLETE]"
)

[COPY] canonical full-gene backbone
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad
->
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_based_annotation.building.h5ad
[COPY COMPLETE]
[ANNOTATION PATCH COMPLETE]


In [13]:
# =========================
# 12) VALIDATE + ATOMICALLY PUBLISH
# =========================

check = sc.read_h5ad(
    ANNOTATED_BUILDING_H5AD,
    backed="r",
)

try:
    if check.shape != (
        EXPECTED_N_CELLS,
        EXPECTED_N_GENES,
    ):
        raise AssertionError(
            "Annotated H5AD shape changed."
        )

    if "counts" not in check.layers:
        raise AssertionError(
            "Annotated H5AD lost layers['counts']."
        )

    for column in (
        ANNOTATION_COL,
        LINEAGE_COL,
        CONFIDENCE_COL,
        MALIGNANCY_COL,
        CAUTION_COL,
        NOTE_COL,
    ):
        if column not in check.obs.columns:
            raise AssertionError(
                f"Annotated H5AD is missing {column!r}."
            )


    final_cluster_counts = (
        check.obs[
            CLUSTER_KEY
        ]
        .astype(str)
        .value_counts()
    )


    if final_cluster_counts.shape[0] != EXPECTED_N_CLUSTERS:
        raise AssertionError(
            "Annotated H5AD lost the canonical 18-cluster partition."
        )


    final_annotation_counts = (
        check.obs[
            ANNOTATION_COL
        ]
        .astype(str)
        .value_counts()
    )


    if final_annotation_counts.shape[0] != EXPECTED_N_CLUSTERS:
        raise AssertionError(
            "Annotated H5AD does not contain 18 annotation labels."
        )


    observed_datasets = set(
        check.obs[
            "core_dataset"
        ]
        .astype(str)
        .unique()
    )


    if observed_datasets != {
        "DS1_GSE103224",
        "DS2_GSE141383",
        "DS3_GSE182109",
        "DS4_GSE278450",
    }:
        raise AssertionError(
            "Final dataset nomenclature changed."
        )


    print(
        "[VALIDATED BUILD]"
    )

    print(
        "shape:",
        check.shape,
    )

    print(
        "counts layer:",
        "counts"
        in check.layers,
    )

    print(
        "canonical clusters:",
        final_cluster_counts.shape[0],
    )

    print(
        "annotation labels:",
        final_annotation_counts.shape[0],
    )

finally:
    check.file.close()


# Remove old compatibility alias first so it cannot keep an obsolete
# multi-gigabyte inode alive after replacement.
if ANNOTATED_H5AD_ALIAS.exists():
    ANNOTATED_H5AD_ALIAS.unlink()


if ANNOTATED_H5AD.exists():
    ANNOTATED_H5AD.unlink()


os.replace(
    ANNOTATED_BUILDING_H5AD,
    ANNOTATED_H5AD,
)


print(
    "[PUBLISHED]"
)

print(
    ANNOTATED_H5AD
)


alias_created = False

try:
    os.link(
        ANNOTATED_H5AD,
        ANNOTATED_H5AD_ALIAS,
    )

    alias_created = True

    print(
        "[HARD-LINK ALIAS CREATED]"
    )

    print(
        ANNOTATED_H5AD_ALIAS
    )

except OSError as error:
    print(
        "[INFO] Hard-link alias was not created:"
    )

    print(
        repr(
            error
        )
    )

    print(
        "Canonical annotated H5AD remains:"
    )

    print(
        ANNOTATED_H5AD
    )

[VALIDATED BUILD]
shape: (351427, 15176)
counts layer: True
canonical clusters: 18
annotation labels: 18
[PUBLISHED]
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_based_annotation.h5ad
[HARD-LINK ALIAS CREATED]
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation.h5ad


In [14]:
# =========================
# 13) FINAL MANIFEST / COMPLETION
# =========================

manifest_items = [
    (
        "annotated_h5ad",
        ANNOTATED_H5AD,
    ),
    (
        "annotated_h5ad_alias",
        ANNOTATED_H5AD_ALIAS,
    ),
    (
        "annotation_tsv",
        ANNOTATION_TSV,
    ),
    (
        "annotation_evidence_tsv",
        ANNOTATION_EVIDENCE_TSV,
    ),
    (
        "annotation_counts_tsv",
        ANNOTATION_COUNTS_TSV,
    ),
    (
        "annotation_composition_tsv",
        ANNOTATION_COMPOSITION_TSV,
    ),
    (
        "annotation_bias_tsv",
        ANNOTATION_BIAS_TSV,
    ),
    (
        "annotation_decision_report_tsv",
        ANNOTATION_DECISION_REPORT_TSV,
    ),
    (
        "annotation_marker_panel_presence_tsv",
        PANEL_PRESENCE_TSV,
    ),
    (
        "figures_dir",
        FIG_DIR,
    ),
]


manifest = pd.DataFrame(
    [
        {
            "item":
                name,

            "path":
                str(
                    path
                ),

            "exists":
                (
                    Path(
                        path
                    ).is_dir()
                    if name
                    == "figures_dir"
                    else Path(
                        path
                    ).exists()
                ),
        }
        for name, path
        in manifest_items
    ]
)


write_tsv_replace(
    manifest,
    ANNOTATION_OUTPUT_MANIFEST_TSV,
)

display(
    manifest
)


if manifest.shape != (
    10,
    3,
):
    raise AssertionError(
        "Output manifest shape changed."
    )


print(
    "\nDONE — canonical historical marker annotation reproduced "
    "with memory-safe execution."
)

print(
    "Annotation table:",
    annotation_table.shape,
)

print(
    "Evidence table:",
    ann.shape,
)

print(
    "Bias table:",
    bias.shape,
)

print(
    "Cell-count table:",
    counts.shape,
)

print(
    "Annotated composition:",
    comp_annot.shape,
)

print(
    "Decision report:",
    decision_report.shape,
)

print(
    "Annotated H5AD:",
    ANNOTATED_H5AD,
)

print(
    "Alias created:",
    alias_created,
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\annotation_marker_based_HVG8000_r0_6_output_manifest.tsv shape=(10, 3)


,item,path,exists
0,annotated_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
1,annotated_h5ad_alias,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
2,annotation_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
3,annotation_evidence_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
4,annotation_counts_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
5,annotation_composition_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
6,annotation_bias_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
7,annotation_decision_report_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
8,annotation_marker_panel_presence_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
9,figures_dir,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True



DONE — canonical historical marker annotation reproduced with memory-safe execution.
Annotation table: (18, 6)
Evidence table: (18, 53)
Bias table: (18, 29)
Cell-count table: (18, 8)
Annotated composition: (2538, 10)
Decision report: (15, 2)
Annotated H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_based_annotation.h5ad
Alias created: True


## Expected successful completion

```text
Annotation table:       (18, 6)
Evidence table:         (18, 53)
Bias table:             (18, 29)
Cell-count table:       (18, 8)
Annotated composition:  (2538, 10)
Decision report:        (15, 2)

plot marker genes present: 59
plot marker genes missing: []

canonical clusters:     18
annotation labels:      18
```

Primary downstream H5AD:

```text
data/processed/integrated/
GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_based_annotation.h5ad
```

The compatibility name
`GBM_core_4datasets_fullgene_marker_based_annotation.h5ad` is created as a
filesystem hard link when supported, so it does not consume another full
multi-gigabyte copy.

No full 351,427 × 15,176 expression matrix is materialized in RAM.